# Neural Network Model - MNIST (CNN)

In [ ]:
%pip install --quiet mlflow==3.6.0 tensorflow==2.20.0 scikit-learn==1.8.0
dbutils.library.restartPython()

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping
import mlflow
import mlflow.tensorflow
from mlflow.models import infer_signature
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import logging
import random

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Get volume_path from DAB parameter (or use default)
seed = int(dbutils.widgets.get('random_state'))
experiment_name = dbutils.widgets.get('experiment_name')
try:
    volume_path = dbutils.widgets.get('volume_path')
except:
    volume_path = '/Volumes/ai_ml_in_practice/mnist_data/processed'

print(f'Volume path: {volume_path}\n')

seed = 15
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)

# Load MNIST images and test data
x_train = np.load(os.path.join(volume_path, 'x_train_images.npy'))
y_train = np.load(os.path.join(volume_path, 'y_train_images.npy'))
x_test = np.load(os.path.join(volume_path, 'x_test_images.npy'))
y_test = np.load(os.path.join(volume_path, 'y_test_images.npy'))

# Reshape for CNN: (samples, 28, 28, 1)
x_train = x_train[..., np.newaxis]
x_test = x_test[..., np.newaxis]

print(f'CNN input shapes - train: {x_train.shape}, test: {x_test.shape}')

mlflow.set_registry_uri('databricks-uc')
mlflow.set_experiment(experiment_name)

In [ ]:
# Build CNN model
model = keras.Sequential([
    keras.Input(shape=(28, 28, 1)),
    layers.Conv2D(8, (3, 3), activation='relu',),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(16, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print('Model built')

In [ ]:
# Train with MLflow
with mlflow.start_run(run_name='cnn_mnist') as run:
    mlflow.log_param('model', 'cnn')
    mlflow.log_param('epochs', 10)
    mlflow.log_param('batch_size', 32)
    
    history = model.fit(
        x_train, y_train,
        validation_split=0.2,
        epochs=10,
        batch_size=32,
        verbose=0,
        callbacks=[EarlyStopping(monitor='val_loss', patience=2)]
    )
    run_id = run.info.run_id
    
    y_pred_proba = model.predict(x_test, verbose=0)
    y_pred = np.argmax(y_pred_proba, axis=1)
    
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    auc = roc_auc_score(y_test, y_pred_proba, multi_class='ovr', average='weighted')
    
    mlflow.log_metric('accuracy', acc)
    mlflow.log_metric('f1', f1)
    mlflow.log_metric('auc', auc)
    
    signature = infer_signature(x_test, y_pred_proba)
    mlflow.tensorflow.log_model(model, 'model', signature=signature)
    mlflow.set_tag('dataset', 'mnist')
    
    print(f'CNN - Accuracy: {acc:.4f}, F1: {f1:.4f}')

try:
    dbutils.jobs.taskValues.set(key='neural_run_id', value=run_id)
except NameError:
    pass